# Project 4 — Quranic Semantic Similarity & Theme Discovery

**Judul:** Mencari Similarity Antar Ayat Al-Qur'an dengan Embedding Transformer, Clustering, dan LLM.

Notebook ini menjalankan eksperimen pada seluruh ayat yang disediakan oleh Al Quran Cloud API. Angka, tabel, dan grafik hanya berasal dari data API dan hasil komputasi pada saat notebook dijalankan; tidak ada hasil eksperimen contoh.

## Latar belakang dan rumusan masalah

Terjemahan ayat dapat dibandingkan secara komputasional untuk menemukan kedekatan representasi semantik, tetapi kedekatan tersebut tidak membuktikan bahwa dua ayat memiliki makna identik. Penelitian ini menggunakan embedding multilingual untuk pencarian tetangga semantik dan clustering, kemudian Gemini membantu merangkum tema berdasarkan teks yang disajikan.

**Pertanyaan penelitian:** (1) Ayat dan pasangan ayat mana yang memiliki cosine similarity embedding tertinggi? (2) Bagaimana hasil evaluasi K-Means untuk beberapa nilai K dan apa karakteristik lima cluster pada K=5? (3) Tema apa yang dapat diringkas LLM dari sampel sistematis anggota setiap cluster, dan apa batas interpretasinya?

**Tujuan:** mengambil dan memvalidasi dataset API; mengukur kemiripan teks terjemahan; membandingkan konfigurasi clustering; menyajikan lima kelompok yang diwajibkan; dan meminta LLM menginterpretasikan tema tanpa menganggapnya sebagai tafsir resmi.

**Batasan:** teks utama adalah edisi terjemahan `id.indonesian`, bukan teks Arab. Model bahasa umum dapat melewatkan nuansa terjemahan, konteks ayat, sebab turun, dan relasi tafsir. Hasil similarity merupakan ukuran representasi model, bukan penilaian teologis.

## Research pipeline dan alasan metode

Alur eksperimen: akuisisi API → validasi → EDA → preprocessing konservatif → embedding → similarity dan ranking → evaluasi clustering → tema LLM → interpretasi. Dataset penuh diminta melalui endpoint edisi API; kedua respons mentah disimpan tanpa ditulis ulang. Ayat disejajarkan berdasarkan nomor global dan diperiksa terhadap identitas surah/ayat.

Model utama `paraphrase-multilingual-MiniLM-L12-v2` dipilih karena mendukung banyak bahasa termasuk bahasa Indonesia dan lebih ringan untuk eksperimen lokal daripada kandidat multilingual MPNet. Ini adalah pemilihan berdasarkan cakupan bahasa dan resource, bukan klaim akurasi tertinggi pada ayat Al-Qur'an. Teks dinormalisasi secara terbatas (Unicode/spasi); stemming dan penghapusan stopword tidak dilakukan.

Dosen melarang cosine similarity TF-IDF. Setelah klarifikasi, cosine similarity atas embedding Transformer diperbolehkan; vektor yang dibandingkan adalah embedding kontekstual 384 dimensi, bukan bobot TF-IDF. K-Means diuji untuk K=3..8 memakai Silhouette (sampel paling banyak 2.000 ayat), Davies-Bouldin, dan Calinski-Harabasz. K=5 dipakai untuk memenuhi persyaratan lima cluster, bukan karena hasilnya pasti paling koheren. Ukuran cluster dan rata-rata pasangan cosine internal dilaporkan terpisah.

Gemini hanya membantu interpretasi tema; setiap cluster diproses terpisah dan prompt/respons disimpan. Untuk membatasi ukuran permintaan, tiap cluster dianalisis melalui sampel sistematis maksimal 30 ayat yang mencakup rentang cosine similarity ke centroid. Hasilnya tidak mewakili pemeriksaan semua anggota. Jika API key belum disetel, notebook menandai tahap sebagai belum terkonfigurasi dan tidak mengarang tema.

## 1. Persiapan lingkungan dan folder proyek

**Tujuan:** memastikan notebook mengimpor pipeline proyek tanpa bergantung pada folder kerja tertentu. **Metode:** mencari modul `quran_similarity.py` pada folder aktif atau subfolder `Project_4`. **Proses:** menyiapkan import dan menampilkan lokasi proyek. **Output yang diharapkan:** lokasi folder proyek dan modul pipeline berhasil dimuat.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / "Project_4", Path.cwd().parent / "Project_4"]
PROJECT_DIR = next((path.resolve() for path in candidates if (path / "quran_similarity.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("quran_similarity.py tidak ditemukan; buka notebook dari workspace atau folder Project_4.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import quran_similarity as qs

print(f"Folder proyek: {PROJECT_DIR}")
print(f"Python: {sys.version.split()[0]}")

**Interpretasi aktual:** notebook ini berjalan dengan Python 3.11.9 dan memuat modul dari folder `Project_4` pada workspace. Versi tersebut mencatat kondisi runtime; hasil penelitian tetap berasal dari data API dan eksperimen di bawah.

## 2. Data acquisition

**Tujuan:** memperoleh teks Arab dan terjemahan Indonesia untuk seluruh surah dari sumber yang ditentukan. **Metode:** endpoint edisi penuh Al Quran Cloud API (`quran-uthmani` dan `id.indonesian`), yang mengembalikan ayat di dalam daftar surah sehingga pagination tidak diperlukan untuk respons ini. **Proses:** unduh respons jika cache raw belum ada; jika cache tersedia, gunakan cache. File JSON raw disimpan apa adanya dan metadata edisi dicatat terpisah. **Output yang diharapkan:** dua payload API dengan informasi edisi dan ayat.

In [ ]:
arabic_payload, translation_payload = qs.acquire_dataset(force_download=False)
display(pd.DataFrame([
    {"edition": payload["data"]["edition"]["identifier"],
     "name": payload["data"]["edition"]["name"],
     "surah_returned": len(payload["data"]["surahs"]),
     "ayah_returned": sum(len(surah["ayahs"]) for surah in payload["data"]["surahs"])}
    for payload in (arabic_payload, translation_payload)
]))

**Interpretasi aktual:** kedua edisi API yang dipilih (`quran-uthmani` dan `id.indonesian`) masing-masing mengembalikan 114 surah dan 6.236 ayat pada run ini. Jika sumber API berubah atau gagal, error ditampilkan; data pengganti sintetis tidak digunakan. Berkas raw yang sudah ada dapat dipakai ulang agar tidak diunduh pada setiap eksekusi.

## 3. Validasi dan pembentukan dataset

**Tujuan:** memastikan setiap terjemahan terpasang pada teks Arab dan identitas ayat yang sesuai. **Metode:** pencocokan nomor ayat global, pemeriksaan jumlah 114 surah/6.236 ayat, nomor ayat dalam surah, duplikasi, teks kosong, dan missing values. **Proses:** gabungkan dua edisi, simpan CSV terstruktur dan laporan validasi. Bila hitungan tidak sesuai, raw dipertahankan dan proses dihentikan agar penyebabnya dapat diperiksa. **Output yang diharapkan:** tabel validasi dan dataset yang tervalidasi.

In [ ]:
dataset, validation_report = qs.validate_and_prepare(arabic_payload, translation_payload)
display(pd.DataFrame([{"check": key, "result": value} for key, value in validation_report.items()]))
display(dataset.head(3))

**Interpretasi aktual:** 114 surah dan 6.236 ayat berhasil disejajarkan; pada laporan run ini tidak ada mismatch nomor/lokasi/juz, missing value, teks kosong, maupun duplikasi identitas, dan semua 30 juz tercakup. Jika jumlah API di run lain berbeda dari 6.236, jangan memperbaiki dengan menghapus atau menambah data secara otomatis; selidiki laporan sebelum analisis.

## 4. Exploratory Data Analysis

**Tujuan:** memahami cakupan, distribusi, dan kualitas dataset sebelum menghitung embedding. **Metode:** hitung ayat per surah/juz, panjang teks terjemahan, missing values, dan grafik bar/histogram. **Proses:** semua statistik dihitung dari dataset hasil validasi. **Output yang diharapkan:** empat grafik EDA tersimpan di `visualizations/` dan ringkasan aktual ditampilkan.

In [ ]:
eda_paths = qs.run_eda(dataset)
print("Path grafik EDA aktual:")
for name, path in eda_paths.items():
    print(f"- {name}: {path}")
print("Missing values per kolom:")
display(dataset.isna().sum().rename("missing_count").to_frame())
print("Ringkasan panjang terjemahan:")
display(dataset["text_indonesian"].str.len().describe().to_frame("character_count"))

**Interpretasi aktual:** dataset berisi 6.236 ayat, median panjang terjemahan 149 karakter, dan tidak ada missing value pada kolom yang diperiksa. Gunakan grafik aktual untuk mengulas distribusi per surah/juz; jumlah ayat maupun panjang teks tidak menunjukkan tema ayat.

## 5. Text preprocessing

**Tujuan:** menyiapkan teks input embedding tanpa menghilangkan informasi semantik. **Metode:** normalisasi Unicode NFC, karakter nonteks yang tidak dapat dicetak, dan spasi berulang; tidak ada stemming atau penghapusan stopword. **Proses:** simpan terjemahan asli dan kolom preprocessing secara terpisah, lalu periksa beberapa perbandingan. **Output yang diharapkan:** CSV preprocessing dan tidak ada teks input kosong.

In [ ]:
preprocessed = qs.prepare_embedding_texts(dataset)
display(preprocessed[["surah_name_en", "ayah_number", "text_indonesian", "text_indonesian_preprocessed"]].head(8))
print(f"Jumlah input preprocessing kosong: {int(preprocessed['text_indonesian_preprocessed'].eq('').sum())}")

**Interpretasi aktual:** tidak ada teks terjemahan yang kosong setelah preprocessing pada run ini. Perbandingan contoh memperlihatkan teks asli tetap tersedia dan teks embedding hanya melalui normalisasi konservatif; stemming maupun stopword removal tidak diterapkan.

## 6. Semantic embedding

**Tujuan:** mengubah setiap terjemahan ayat menjadi vektor yang dapat dibandingkan secara semantik. **Metode:** Sentence Transformer multilingual dengan representasi ternormalisasi dan batch processing. Kandidat MPNet memiliki representasi lebih besar, tetapi memerlukan resource lebih tinggi; MiniLM dipilih untuk kompromi cakupan bahasa dan ukuran. Tabel berikut membandingkan kelayakan secara kualitatif, bukan skor akurasi pada dataset ini. **Proses:** model yang dipilih dimuat pada GPU jika tersedia, lalu embedding disimpan agar tidak dihitung ulang. **Output yang diharapkan:** nama model, perangkat, dan dimensi embedding aktual.

In [ ]:
model_comparison = pd.DataFrame([
    {"model": "paraphrase-multilingual-MiniLM-L12-v2", "language/STS": "Multilingual, mendukung Indonesia; ditujukan untuk semantic search/similarity", "dimension": 384, "resource_fit": "Lebih ringan; dipilih untuk eksperimen lokal"},
    {"model": "paraphrase-multilingual-mpnet-base-v2", "language/STS": "Multilingual, mendukung Indonesia; kandidat semantic similarity", "dimension": 768, "resource_fit": "Lebih besar; tidak dipilih tanpa evaluasi langsung pada dataset"},
])
display(model_comparison)
print("Catatan: tabel ini adalah perbandingan spesifikasi/kelayakan, bukan benchmark akurasi empiris antarmodel pada ayat Al-Qur'an.")

**Interpretasi:** MiniLM dipakai sebagai baseline yang lebih hemat resource; dimensi aktual dan ketersediaan perangkat dicetak oleh eksekusi model. Perbandingan ini tidak membuktikan bahwa model pilihan lebih akurat atau memahami konteks tafsir. Terjemahan dapat memiliki diksi dan struktur yang berbeda dari bahasa Arab.

**Tujuan:** menghasilkan satu embedding kontekstual per ayat. **Metode:** `SentenceTransformer.encode` dengan embedding ternormalisasi dan batch 64; CUDA digunakan jika tersedia, selain itu CPU. **Proses:** muat cache jika sesuai dengan model dan jumlah ayat, atau hitung ulang dan simpan `.npy` beserta metadata. **Output yang diharapkan:** matriks berukuran jumlah ayat × dimensi model.

In [ ]:
embeddings, embedding_model = qs.load_or_create_embeddings(preprocessed)
print(f"Bentuk matriks embedding: {embeddings.shape}")
print(f"Tipe data: {embeddings.dtype}; seluruh nilai finite: {bool(pd.notna(embeddings).all())}")
print(f"Dimensi dari model: {embeddings.shape[1]}")

**Interpretasi aktual:** model menghasilkan 6.236 embedding berdimensi 384 pada CUDA; seluruh nilai terdeteksi finite. Karena embedding dinormalisasi, dot product ekuivalen dengan cosine similarity. Similarity tetap merupakan kedekatan representasi model dan memiliki keterbatasan konteks.

## 7. Semantic similarity dan ranking

**Tujuan:** menjawab dua pertanyaan ranking yang berbeda. **Metode:** cosine similarity pada embedding Transformer yang telah dinormalisasi dan pencarian nearest neighbors; tidak digunakan TF-IDF. **Proses:** untuk setiap ayat cari lima ayat lain tanpa memasukkan sumber sebagai kandidat dirinya; hitung pula pasangan global tanpa duplikasi A–B/B–A. Lima ayat sumber tambahan diurutkan menurut rata-rata similarity lima tetangga masing-masing. Heatmap memakai sampel sistematis maksimal 40 ayat agar tidak membuat matriks seluruh dataset. **Output yang diharapkan:** CSV ranking, grafik, dan tabel aktual.

In [ ]:
similarity_results = qs.calculate_similarity(preprocessed, embeddings, top_k=5, global_pair_count=100)
top5_per_verse = similarity_results["nearest_verses"]
global_pairs = similarity_results["global_pairs"]
top5_source_verses = similarity_results["top_verses"]
assert not (top5_per_verse["source_global_number"] == top5_per_verse["match_global_number"]).any()
assert len(global_pairs) == len(global_pairs[["global_number_1", "global_number_2"]].drop_duplicates())
assert not (global_pairs["global_number_1"] >= global_pairs["global_number_2"]).any()
print("Lima ayat sumber dengan rata-rata similarity top-5 tertinggi:")
display(top5_source_verses)
print("Top-5 tetangga untuk ayat sumber dengan ranking tertinggi:")
display(top5_per_verse[top5_per_verse["source_global_number"] == int(top5_source_verses.iloc[0]["source_global_number"])])
print(f"Pasangan top-5 dengan teks terjemahan identik: {int(top5_per_verse['translation_exact_match'].sum()):,} dari {len(top5_per_verse):,}")
print("20 pasangan global unik teratas; kolom exact_match menandai terjemahan yang identik:")
display(global_pairs.head(20)[["rank", "surah_name_1", "ayah_number_1", "text_1", "surah_name_2", "ayah_number_2", "text_2", "similarity", "translation_exact_match"]])

**Interpretasi aktual:** pada eksekusi ini, lima ayat sumber teratas adalah Al-Mursalaat 34, 37, 40, 45, dan 47 dengan rata-rata top-5 similarity 1,0 karena terjemahannya berulang identik ("Kecelakaan yang besarlah pada hari itu bagi orang-orang yang mendustakan."). Pasangan global teratas juga memuat terjemahan identik, misalnya Alif Laam Miim. Kolom `translation_exact_match` membantu mengenali kecocokan literal tanpa menghapusnya. Karena itu nilai 1,0 di sini tidak boleh digeneralisasi sebagai kesamaan semantik kontekstual. Tabel pertama memilih lima ayat sumber dengan rata-rata similarity ke lima tetangga terdekat tertinggi; tabel pasangan global mengurutkan pasangan unik dari seluruh dataset. Keduanya memiliki tujuan berbeda. Periksa teks dan identitas ayat sebelum menarik kesimpulan.

## 8. Clustering dan evaluasi

**Tujuan:** membentuk lima kelompok ayat dan mengukur kualitas konfigurasi clustering. **Metode:** K-Means pada embedding ternormalisasi untuk K=3,4,5,6,7,8; Silhouette dihitung pada sampel maksimal 2.000 ayat untuk efisiensi, sedangkan Davies-Bouldin dan Calinski-Harabasz menggunakan data penuh. Rata-rata cosine internal tiap cluster dihitung eksak dari embedding anggotanya. **Proses:** simpan evaluasi, label K=5, ukuran cluster, dan grafik PCA. **Output yang diharapkan:** tabel metrik, lima cluster yang terbentuk, serta grafik evaluasi dan proyeksi.

In [ ]:
cluster_evaluation, clustered = qs.run_clustering(preprocessed, embeddings)
clustered_with_centroid = qs.attach_centroid_similarity(clustered, embeddings)
display(cluster_evaluation.sort_values("k"))
cluster_summary = clustered_with_centroid.groupby("cluster").agg(
    member_count=("global_number", "size"),
    mean_cosine_to_centroid=("cosine_similarity_to_cluster_centroid", "mean"),
).reset_index()
internal_similarity = pd.read_csv(qs.CLUSTERING_DIR / "cluster_summary_k5.csv")
cluster_summary = cluster_summary.merge(internal_similarity, on=["cluster", "member_count"], how="left")
display(cluster_summary)
print(f"Label cluster unik aktual: {sorted(clustered_with_centroid['cluster'].unique().tolist())}")

**Interpretasi aktual:** pada run ini Silhouette tertinggi berasal dari K=5 (0,03948), tetapi nilainya rendah; Davies-Bouldin terendah berasal dari K=4 (3,45047), sedangkan Calinski-Harabasz tertinggi dari K=3 (332,38). Metrik tidak menunjuk satu konfigurasi yang unggul pada semua ukuran. Untuk K=5, rata-rata cosine internal bervariasi sekitar 0,270–0,568 dan Silhouette tetap rendah, sehingga pemisahan kelompok lemah dan tema harus dianggap eksploratif, bukan kelompok semantik yang telah tervalidasi. K=5 digunakan untuk memenuhi ketentuan jumlah cluster; ini bukan klaim lima kelompok terbesar atau paling koheren. Ukuran anggota dan rata-rata similarity internal adalah ringkasan berbeda; baca ayat anggota dan konteks sebelum memberi tema.

## 9. LLM-based theme discovery

**Tujuan:** meminta Gemini merangkum tema dari teks anggota cluster, bukan menggantikan perhitungan similarity atau menetapkan tafsir. **Metode:** satu permintaan terstruktur JSON per cluster, berisi referensi, terjemahan, dan similarity centroid dari sampel sistematis maksimal 30 ayat yang mencakup rentang nilai similarity. Sampel membatasi ukuran prompt tetapi tidak meninjau semua anggota cluster. Respons dan prompt aktual dicache. **Proses:** key dibaca dari variabel environment atau file `.env\gemini.env`; key tidak perlu ditulis di notebook. Jika belum tersedia, fungsi mencatat status konfigurasi yang terblokir dan notebook tetap berjalan tanpa tema rekaan. **Output yang diharapkan:** lima respons aktual Gemini atau status belum terkonfigurasi.

In [ ]:
theme_status = qs.discover_themes(clustered_with_centroid)
display(pd.DataFrame([theme_status]))
theme_results_path = qs.THEME_DIR / "theme_analysis.json"
if theme_status.get("status") == "completed" and theme_results_path.is_file():
    import json
    theme_results = json.loads(theme_results_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([
        {"cluster": item["cluster"], **item["response"]}
        for item in theme_results
    ]))
else:
    print("Tema belum tersedia sebagai hasil eksperimen aktual. Setelah GEMINI_API_KEY dikonfigurasi, jalankan ulang cell ini.")

**Interpretasi aktual:** Gemini berhasil mengembalikan lima tema berbasis sampel: (1) Risalah Ilahi dan Respons Manusia; (2) Konsekuensi Iman dan Kekafiran serta Gambaran Akhirat; (3) Kekuasaan Ilahi, Penciptaan, dan Takdir Manusia; (4) Keesaan Allah, Hari Kiamat, dan Konsekuensi Amal; (5) Konsekuensi Kufur dan Penolakan Petunjuk Ilahi. Setiap cluster diwakili 30 ayat dari 586–2.050 anggota, jadi tema tersebut belum tentu mewakili seluruh anggotanya. Tinjau ayat representatif dan keterbatasan dari respons tersimpan. Tema LLM bukan tafsir resmi dan similarity tinggi tidak membuktikan ayat identik.

## 10. Pembahasan, keterbatasan, dan kesimpulan

**Tujuan:** merangkum bukti numerik aktual dan batas inferensi. **Metode:** tampilkan jumlah data tervalidasi, ringkasan metrik clustering, ukuran cluster, hasil pasangan global, dan status analisis tema yang tersimpan. **Proses:** cell berikut hanya membaca objek hasil eksperimen sebelumnya dan tidak membentuk kesimpulan otomatis dari data yang belum tersedia. **Output yang diharapkan:** inventaris hasil aktual yang dapat digunakan untuk menulis diskusi dan kesimpulan penelitian.

In [ ]:
print(f"Validasi dataset: {'lulus' if validation_report['validation_passed'] else 'gagal'}")
print(f"Jumlah surah: {validation_report['translation_surah_count']}; jumlah ayat: {validation_report['translation_ayah_count']:,}")
print(f"Pasangan global tersimpan: {len(global_pairs)}; similarity maksimum yang ditampilkan: {global_pairs['similarity'].max():.6f}")
print(f"Status analisis tema: {theme_status['status']}")
display(cluster_evaluation.sort_values("silhouette_sampled", ascending=False).head(3))
display(cluster_summary.sort_values("mean_pairwise_cosine", ascending=False))
print("Gunakan nilai dan tabel aktual di atas untuk pembahasan; jangan menyamakan peringkat metrik dengan validitas tafsir.")

**Panduan kesimpulan:** jawab pertanyaan penelitian dengan nilai similarity, tabel metrik, ukuran/koherensi internal cluster, dan respons Gemini aktual bila tersedia. Laporkan bahwa model bekerja pada terjemahan Indonesia, clustering K=5 ditentukan oleh kebutuhan tugas, Silhouette menggunakan sampling, dan konteks linguistik/tafsir tidak dimodelkan sepenuhnya. Jangan menyatakan kelompok valid semantik hanya karena K-Means menghasilkan lima label. Bila Gemini belum dikonfigurasi, nyatakan tema sebagai pekerjaan yang belum dapat diselesaikan, bukan mengisi kesimpulan dengan perkiraan.

## Referensi

1. Al Quran Cloud. *API Documentation*. https://alquran.cloud/api (diakses sesuai tanggal pada metadata data).
2. Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. EMNLP-IJCNLP. https://doi.org/10.18653/v1/D19-1410.
3. Hugging Face model cards: [paraphrase-multilingual-MiniLM-L12-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) dan [paraphrase-multilingual-mpnet-base-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2).
4. Rousseeuw, P. J. (1987). *Silhouettes: A graphical aid to the interpretation and validation of cluster analysis*. Journal of Computational and Applied Mathematics, 20, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7.
5. Davies, D. L. & Bouldin, D. W. (1979). *A Cluster Separation Measure*. IEEE Transactions on Pattern Analysis and Machine Intelligence, PAMI-1(2), 224–227. https://doi.org/10.1109/TPAMI.1979.4766909.
6. Caliński, T. & Harabasz, J. (1974). *A dendrite method for cluster analysis*. Communications in Statistics, 3(1), 1–27. https://doi.org/10.1080/03610927408827101.
7. Google AI for Developers. *Gemini API structured output*. https://ai.google.dev/gemini-api/docs/structured-output.

Catat versi paket, tanggal eksekusi, metadata API, dan setiap kegagalan runtime ketika menyusun laporan akhir.